# Preprocessing Data: retail_store_sales.csv
## Untuk Analisis Apriori dan FP-Growth

Notebook ini melakukan preprocessing dataset `retail_store_sales.csv` agar sesuai dengan format dataset contoh `UTS_Apriori dan FP-Growth.xlsx - Datasets`.

**Langkah-langkah Preprocessing:**
1. Import Library
2. Load Dataset
3. Eksplorasi Data Awal (EDA)
4. Data Cleaning (Menangani Missing Values, Duplikat)
5. Transformasi Data ke Format Transaksi (Market Basket)
6. Export Hasil Preprocessing

## 1. Import Library

In [ ]:
%pip install pandas openpyxl mlxtend

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimport!')

## 2. Load Dataset

In [ ]:
# Load dataset retail_store_sales
df = pd.read_csv('retail_store_sales.csv')

print(f'Jumlah baris: {df.shape[0]}')
print(f'Jumlah kolom: {df.shape[1]}')
print(f'\nKolom dataset:')
print(df.columns.tolist())
df.head(10)

## 3. Eksplorasi Data Awal (EDA)

In [ ]:
# Informasi dataset
print('=== Info Dataset ===')
df.info()

In [ ]:
# Statistik deskriptif
print('=== Statistik Deskriptif ===')
df.describe()

In [ ]:
# Cek missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct.round(2)})
print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Cek duplikat
print('=== Duplikat ===')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

In [ ]:
# Cek nilai unik untuk kolom kategorikal
print('=== Nilai Unik ===')
print(f"Jumlah Transaction ID unik: {df['Transaction ID'].nunique()}")
print(f"Jumlah Customer ID unik: {df['Customer ID'].nunique()}")
print(f"\nKategori produk ({df['Category'].nunique()} kategori):")
print(df['Category'].value_counts())
print(f"\nJumlah Item unik: {df['Item'].nunique()}")
print(f"\nPayment Method:")
print(df['Payment Method'].value_counts())
print(f"\nLocation:")
print(df['Location'].value_counts())
print(f"\nDiscount Applied:")
print(df['Discount Applied'].value_counts())

## 4. Data Cleaning

In [ ]:
# Simpan ukuran awal
print(f'Ukuran dataset awal: {df.shape}')

# 4.1 Hapus baris duplikat
df_clean = df.drop_duplicates()
print(f'Setelah hapus duplikat: {df_clean.shape}')

# 4.2 Hapus baris yang memiliki missing value pada kolom Item
# (kolom Item penting karena akan digunakan untuk analisis asosiasi)
print(f"\nMissing values pada kolom 'Item' sebelum cleaning: {df_clean['Item'].isnull().sum()}")
df_clean = df_clean.dropna(subset=['Item'])
print(f"Setelah hapus missing pada 'Item': {df_clean.shape}")

# 4.3 Hapus baris yang memiliki missing value pada kolom Quantity dan Total Spent
print(f"\nMissing values pada 'Quantity': {df_clean['Quantity'].isnull().sum()}")
print(f"Missing values pada 'Total Spent': {df_clean['Total Spent'].isnull().sum()}")
df_clean = df_clean.dropna(subset=['Quantity', 'Total Spent'])
print(f"Setelah hapus missing pada 'Quantity' & 'Total Spent': {df_clean.shape}")

# 4.4 Untuk kolom Discount Applied, isi missing value dengan 'Unknown'
df_clean['Discount Applied'] = df_clean['Discount Applied'].fillna('Unknown')

print(f'\n=== Hasil Cleaning ===')
print(f'Ukuran dataset bersih: {df_clean.shape}')
print(f'Total missing values tersisa: {df_clean.isnull().sum().sum()}')

In [ ]:
# Verifikasi data bersih
print('=== Verifikasi Data Bersih ===')
print(df_clean.isnull().sum())
print(f'\nJumlah duplikat: {df_clean.duplicated().sum()}')
df_clean.head(10)

## 5. Transformasi Data ke Format Transaksi (Market Basket)

Transformasi dataset dari format baris per-item menjadi format market basket seperti pada contoh dataset `UTS_Apriori dan FP-Growth.xlsx - Datasets`, dimana:
- Setiap baris = 1 transaksi
- Kolom `Item(s)` = jumlah item dalam transaksi
- Kolom `Item 1`, `Item 2`, ... = nama item yang dibeli

In [ ]:
# Grouping item berdasarkan Transaction ID
# Setiap transaksi berisi daftar item yang dibeli
transactions = df_clean.groupby('Transaction ID')['Item'].apply(list).reset_index()
transactions.columns = ['Transaction ID', 'Items']

# Hitung jumlah item per transaksi
transactions['Item(s)'] = transactions['Items'].apply(len)

print(f'Jumlah transaksi: {len(transactions)}')
print(f'Rata-rata item per transaksi: {transactions["Item(s)"].mean():.2f}')
print(f'Max item per transaksi: {transactions["Item(s)"].max()}')
print(f'Min item per transaksi: {transactions["Item(s)"].min()}')
print(f'\nDistribusi jumlah item per transaksi:')
print(transactions['Item(s)'].value_counts().sort_index())

transactions.head(10)

In [ ]:
# Transformasi ke format seperti contoh dataset UTS
# Kolom: Item(s), Item 1, Item 2, ..., Item N

max_items = transactions['Item(s)'].max()
print(f'Jumlah kolom item maksimal: {max_items}')

# Buat DataFrame baru dengan format market basket
item_columns = [f'Item {i+1}' for i in range(max_items)]

# Pecah list items menjadi kolom-kolom terpisah
items_expanded = transactions['Items'].apply(pd.Series)
items_expanded.columns = item_columns[:items_expanded.shape[1]]

# Gabungkan dengan kolom Item(s)
df_basket = pd.concat([transactions[['Item(s)']], items_expanded], axis=1)

# Isi NaN dengan string kosong (untuk item yang tidak ada)
df_basket = df_basket.fillna('')

print(f'\nUkuran dataset market basket: {df_basket.shape}')
print(f'Kolom: {df_basket.columns.tolist()}')
df_basket.head(10)

In [ ]:
# Bandingkan dengan format contoh dataset UTS
print('=== Perbandingan Format ===')

# Load contoh dataset UTS
df_contoh = pd.read_csv('UTS_Apriori dan FP-Growth.xlsx - Datasets.csv')
print(f'\nFormat Contoh (UTS):')
print(f'  - Jumlah baris: {df_contoh.shape[0]}')
print(f'  - Jumlah kolom: {df_contoh.shape[1]}')
print(f'  - Kolom: {df_contoh.columns.tolist()[:5]} ...')

print(f'\nFormat Hasil Preprocessing:')
print(f'  - Jumlah baris: {df_basket.shape[0]}')
print(f'  - Jumlah kolom: {df_basket.shape[1]}')
print(f'  - Kolom: {df_basket.columns.tolist()[:5]} ...')

print(f'\n✅ Format sudah sesuai dengan contoh dataset UTS!')

## 6. Export Hasil Preprocessing

In [ ]:
# Export ke CSV
df_basket.to_csv('retail_store_sales_preprocessed.csv', index=False)
print('✅ Dataset berhasil disimpan ke: retail_store_sales_preprocessed.csv')

# Export ke Excel
df_basket.to_excel('retail_store_sales_preprocessed.xlsx', index=False, sheet_name='Datasets')
print('✅ Dataset berhasil disimpan ke: retail_store_sales_preprocessed.xlsx')

# Export data bersih (sebelum transformasi) ke CSV
df_clean.to_csv('retail_store_sales_clean.csv', index=False)
print('✅ Dataset bersih disimpan ke: retail_store_sales_clean.csv')

In [ ]:
# Ringkasan akhir
print('=' * 60)
print('RINGKASAN PREPROCESSING')
print('=' * 60)
print(f'Dataset awal          : {df.shape[0]} baris x {df.shape[1]} kolom')
print(f'Setelah cleaning      : {df_clean.shape[0]} baris x {df_clean.shape[1]} kolom')
print(f'Data dihapus          : {df.shape[0] - df_clean.shape[0]} baris')
print(f'Jumlah transaksi      : {len(df_basket)} transaksi')
print(f'Max item/transaksi    : {df_basket["Item(s)"].max()}')
print(f'Format output         : Item(s), Item 1, Item 2, ..., Item {max_items}')
print(f'\nFile output:')
print(f'  1. retail_store_sales_clean.csv          (data bersih)')
print(f'  2. retail_store_sales_preprocessed.csv   (format market basket)')
print(f'  3. retail_store_sales_preprocessed.xlsx   (format market basket)')
print('=' * 60)